# Credit Card Fraud Detection: End-to-End Machine Learning Workflow
### Detecting 0 (Legitimate) vs 1 (Fraudulent) Financial Transactions

**Author:** ML Engineering Portfolio Project  
**Dataset:** European Credit Card Transactions (September 2013, ULB Machine Learning Group)  

---

### Project Outline
1. **Problem Formulation & Imbalance Challenge**
2. **Exploratory Data Analysis (EDA) & Anomaly Detection**
3. **Data Preprocessing & Preventing Data Leakage**
4. **Class Imbalance Strategy: Cost-Sensitive Weights vs SMOTE**
5. **Model Benchmarking: Logistic Regression, Random Forest, XGBoost**
6. **Rigorous Evaluation: PR-AUC, ROC-AUC, Precision-Recall Trade-offs**
7. **Cost-Sensitive Confusion Matrix & Banking Economics**
8. **Feature Importance & Model Explainability**
9. **Model Serialization (`joblib`) & Inference Verification**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGBOOST = False

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("Libraries loaded successfully.")

## 1. Load Dataset & Schema Verification
The dataset contains 284,807 transactions. Features `V1` through `V28` are numerical principal components resulting from PCA transformation for confidentiality. Only `Time`, `Amount`, and `Class` (target) are un-transformed.

In [ ]:
DATA_PATH = "../data/creditcard.csv"
if not os.path.exists(DATA_PATH):
    # Fallback to sample data if full dataset has not yet been downloaded
    DATA_PATH = "../data/sample_transactions.csv"
    print(f"[NOTE] Full dataset not detected. Using sample file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
display(df.head())

In [ ]:
# Missing values & duplicate checking
print("Missing values per column:", df.isnull().sum().max())
duplicates = df.duplicated().sum()
print(f"Duplicate rows detected: {duplicates:,}")

if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dataset shape after removing duplicates: {df.shape[0]:,}")

## 2. Exploratory Data Analysis (EDA)
Let us explore the target distribution, amount statistics, and the extreme class imbalance.

In [ ]:
class_counts = df["Class"].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100

print(f"Legitimate Transactions (0): {class_counts[0]:,} ({100 - fraud_pct:.3f}%)")
print(f"Fraudulent Transactions (1): {class_counts[1]:,} ({fraud_pct:.3f}%)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(x="Class", data=df, ax=ax[0], palette=["#10b981", "#ef4444"])
ax[0].set_yscale("log")
ax[0].set_title("Class Distribution (Log Scale)")

sns.boxplot(x="Class", y="Amount", data=df, ax=ax[1], palette=["#10b981", "#ef4444"], showfliers=False)
ax[1].set_title("Transaction Amount by Class (Excluding Outliers)")
plt.tight_layout()
plt.show()

## 3. Correlation Analysis: Identifying Fraud Anomaly Drivers
Because V1-V28 are orthogonal PCA components, cross-feature correlation is 0. However, their correlation with the binary `Class` label reveals which latent axes carry the strongest predictive signal for fraud.

In [ ]:
correlations = df.corr()["Class"].sort_values()
top_neg = correlations.head(6)
top_pos = correlations.tail(6).drop(index=["Class"])
top_corrs = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 4))
colors = ["#ef4444" if x < 0 else "#3b82f6" for x in top_corrs.values]
top_corrs.plot(kind="barh", color=colors)
plt.title("Top Correlated Features with Fraud Target (Class)")
plt.xlabel("Correlation Coefficient")
plt.show()

## 4. Preprocessing & Preventing Data Leakage

**Strict Data Leakage Rules:**
1. **Stratified Train-Test Split First**: Splitting before any transformation guarantees the test set remains an untouched representation of real future transactions.
2. **Fit Scaler on Train Only**: `Amount` and `Time` have large ranges and outliers; we use `RobustScaler` (median and IQR), fitting exclusively on `X_train`.
3. **SMOTE Applied Post-Split**: Oversampling generates synthetic observations by interpolating between nearest neighbors. If applied before splitting, neighbor points between future test samples and train samples bleed into each other.

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"].astype(int)

# 1. Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. RobustScaler fitted ONLY on training set
scaler = RobustScaler()
scale_cols = [c for c in ["Amount", "Time"] if c in X_train.columns]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"Train size: {X_train_scaled.shape[0]:,} | Test size: {X_test_scaled.shape[0]:,}")
print(f"Train Frauds: {y_train.sum():,} | Test Frauds: {y_test.sum():,}")

## 5. Handling Imbalance: Cost-Sensitive Weights vs SMOTE
We evaluate two primary strategies:
- `class_weight='balanced'`: Penalizes mistakes on minority frauds inversely proportional to class frequency.
- `SMOTE`: Synthetically generates minority points along line segments joining k-nearest neighbors in training data.

In [ ]:
# Demonstrating SMOTE on X_train only
smote = SMOTE(sampling_strategy=0.20, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"SMOTE Resampled Train: {len(y_train_smote):,} records (Frauds: {y_train_smote.sum():,})")

## 6. Model Training & Comparison
We train and benchmark:
1. **Logistic Regression** (Balanced)
2. **Random Forest** (Balanced Subsample)
3. **XGBoost / Gradient Boosting**

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=12, class_weight="balanced_subsample",
        n_jobs=-1, random_state=42
    )
}

if HAS_XGBOOST:
    imbalance_ratio = float((y_train == 0).sum() / max(y_train.sum(), 1))
    models["XGBoost"] = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=imbalance_ratio, eval_metric="logloss",
        n_jobs=-1, random_state=42
    )

results = {}
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "y_pred": y_pred,
        "y_prob": y_prob,
        "model": clf
    }

comparison_df = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC"]]
display(comparison_df.round(4))

## 7. Precision-Recall Curves & ROC Analysis
In severely imbalanced fraud detection (0.17%), **PR-AUC is the true gold-standard metric**. ROC-AUC can present an overly optimistic picture because the massive volume of True Negatives keeps the False Positive Rate artificially low.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for name in results.keys():
    prec, rec, _ = precision_recall_curve(y_test, results[name]["y_prob"])
    ax[0].plot(rec, prec, label=f"{name} (PR-AUC = {results[name]['PR-AUC']:.3f})")
    
    fpr, tpr, _ = roc_curve(y_test, results[name]["y_prob"])
    ax[1].plot(fpr, tpr, label=f"{name} (ROC-AUC = {results[name]['ROC-AUC']:.3f})")

ax[0].set_title("Precision-Recall Curve (Gold Standard)")
ax[0].set_xlabel("Recall")
ax[0].set_ylabel("Precision")
ax[0].legend()

ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set_title("Receiver Operating Characteristic (ROC)")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend()

plt.tight_layout()
plt.show()

## 8. Confusion Matrix & Banking Economics
In banking fraud engines:
- **False Negative (Missed Fraud):** Cost = full transaction chargeback loss + processing fee + customer loss.
- **False Positive (False Alarm):** Cost = SMS verification fee + minor customer friction.
A lower threshold (e.g. 0.35) can capture significantly more frauds.

In [ ]:
best_name = max(results.keys(), key=lambda k: results[k]["PR-AUC"])
best_cm = confusion_matrix(y_test, results[best_name]["y_pred"])

plt.figure(figsize=(6, 4))
sns.heatmap(best_cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted Legit", "Predicted Fraud"],
            yticklabels=["Actual Legit", "Actual Fraud"])
plt.title(f"Confusion Matrix: {best_name}")
plt.show()

tn, fp, fn, tp = best_cm.ravel()
print(f"True Negatives (Legit Approved):  {tn:,}")
print(f"False Positives (False Alarms):  {fp:,}")
print(f"False Negatives (Missed Frauds): {fn:,}  <-- Most Critical Banking Risk")
print(f"True Positives (Frauds Blocked): {tp:,}")

## 9. Feature Importance & Explainability

In [ ]:
best_clf = results[best_name]["model"]
if hasattr(best_clf, "feature_importances_"):
    importances = pd.Series(best_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10, 4))
    importances.head(10).plot(kind="bar", color="#6366f1")
    plt.title(f"Top 10 Feature Importances ({best_name})")
    plt.ylabel("Relative Importance Score")
    plt.show()

## 10. Model Serialization with Joblib
We serialize the champion model, fitted scaler, and metadata into `models/fraud_detection_model.pkl` for production inference in our Streamlit application.

In [ ]:
os.makedirs("../models", exist_ok=True)
artifact = {
    "model": best_clf,
    "model_name": best_name,
    "scaler": scaler,
    "feature_names": list(X.columns),
    "threshold": 0.50,
    "metrics": comparison_df.loc[best_name].to_dict()
}

output_file = "../models/fraud_detection_model.pkl"
joblib.dump(artifact, output_file)
print(f"Successfully exported model artifact to: {output_file}")